# Ridge model builder (no cross validation)

Builds one Ridge regression per sales target using the **already-selected best parameters**
from the earlier grid search. No cross validation is performed here.

Pipeline per target: train/test split -> `StandardScaler` (fit on train only) -> `Ridge` (fit on train only).

Outputs (all under `MODELS_DIR`, safe to rerun; each run rebuilds the folder from scratch):
* `<target>/scaler.joblib` and `<target>/ridge.joblib` - fitted sklearn objects for each target
* `feature_map.json` - the one feature order every model expects
* `model_metadata.csv` - index = target; columns = per-feature weights, scaler mean/scale, intercept,
  hyperparameters and metrics. Enough to predict without sklearn:
  `y = intercept + sum(weight_f * (x_f - scaler_mean_f) / scaler_scale_f)`

Standardized-feature weights describe associations, not causal importance.

In [1]:
import json
import shutil
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Configuration
Edit paths here only. Model hyperparameters, split settings, and random state come from the parameter CSV.

In [2]:
PROJECT_DIR = Path('/home/lelandmesford/Projects/Predict_Car_Sales')
TARGETS_PATH = PROJECT_DIR / 'car_sales_prediciton_targets.csv'
FEATURES_PATH = PROJECT_DIR / 'car_sales_prediction_features.csv'
PARAMETERS_PATH = PROJECT_DIR / 'best_parameters_2026-10-16.csv'
MODELS_DIR = PROJECT_DIR / 'models'

# Column-name prefixes used in the metadata CSV, so columns are grouped and self-describing.
WEIGHT_PREFIX = 'weight__'
MEAN_PREFIX = 'scaler_mean__'
SCALE_PREFIX = 'scaler_scale__'

## Load and validate data
Dates must be unique, parseable, and identical in both files. No imputation or lagging is done.

In [3]:
targets = pd.read_csv(TARGETS_PATH)
features = pd.read_csv(FEATURES_PATH)


def find_date_column(frame, name):
    """Return the single column that looks like a date and parses fully."""
    candidates = [c for c in frame.columns if 'date' in c.lower() or 'time' in c.lower()]
    valid = [c for c in candidates if pd.to_datetime(frame[c], errors='coerce').notna().all()]
    if len(valid) != 1:
        raise ValueError(f'{name}: ambiguous or invalid date columns {candidates}')
    return valid[0]


def normalise_dates(frame, name):
    col = find_date_column(frame, name)
    out = frame.copy()
    out['date'] = pd.to_datetime(out[col])
    if col != 'date':
        out = out.drop(columns=col)
    if out['date'].duplicated().any():
        raise ValueError(f'{name} has duplicate dates')
    return out


targets = normalise_dates(targets, 'targets')
features = normalise_dates(features, 'features')
if set(targets['date']) != set(features['date']):
    raise ValueError('Target and feature dates differ')

# Columns shared by both files (other than date) are identifiers, not model inputs.
shared = sorted((set(targets.columns) & set(features.columns)) - {'date'})
target_columns = [c for c in targets.columns if c not in {'date', *shared}]
# Feature order is fixed here and is the order every saved model expects.
feature_columns = [c for c in features.columns if c not in {'date', *shared}]

merged = targets[['date', *target_columns]].merge(
    features[['date', *feature_columns]], on='date', how='inner', validate='one_to_one')
if len(merged) != len(targets):
    raise ValueError('Merge dropped rows')
if merged[target_columns + feature_columns].isna().any().any():
    raise ValueError('Missing values found; no imputation is performed')

X = merged[feature_columns]
print(f'{len(merged)} rows, {len(target_columns)} targets, {len(feature_columns)} features')

21 rows, 21 targets, 308 features


## Load best parameters
One row per target. Pandas reads Python `None` as NaN, so NaN is converted back to `None` before use.

In [4]:
params = pd.read_csv(PARAMETERS_PATH).set_index('target')
missing = set(target_columns) - set(params.index)
if missing:
    raise ValueError(f'No saved parameters for targets: {sorted(missing)}')


def clean(value):
    """Convert NaN -> None and numpy scalars -> plain Python types."""
    if pd.isna(value):
        return None
    return value.item() if hasattr(value, 'item') else value


def int_or_none(value):
    value = clean(value)
    return None if value is None else int(value)


# Split settings must be identical across targets so every target shares one train/test split.
split_cols = ['split_test_size', 'split_train_size', 'split_shuffle', 'split_random_state']
if (params.loc[target_columns, split_cols].nunique(dropna=False) > 1).any():
    raise ValueError('Split settings differ between targets')
first = params.loc[target_columns[0]]
SPLIT = {
    'test_size': float(first['split_test_size']),
    'train_size': float(first['split_train_size']),
    'shuffle': bool(first['split_shuffle']),
    'random_state': int_or_none(first['split_random_state']),
}
print('Split settings:', SPLIT)

Split settings: {'test_size': 0.3, 'train_size': 0.7, 'shuffle': True, 'random_state': 42}


## Train/test split
Split row positions once so all targets are trained and evaluated on the same rows.

In [5]:
train_idx, test_idx = train_test_split(np.arange(len(merged)), **SPLIT)
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
print(f'train={len(train_idx)} test={len(test_idx)}')

train=14 test=7


## Fit scaler and Ridge for each target
The scaler learns mean/std from the training rows only (no test leakage). Ridge is fit on the scaled training rows.
A manual NumPy prediction from the saved weights is checked against sklearn to prove the metadata is sufficient.

In [6]:
# Rebuild the models folder from scratch so reruns never leave stale target folders behind.
if MODELS_DIR.exists():
    if MODELS_DIR.resolve().parent != PROJECT_DIR.resolve():
        raise RuntimeError(f'Refusing to delete unexpected path {MODELS_DIR}')
    shutil.rmtree(MODELS_DIR)
MODELS_DIR.mkdir(parents=True)

rows = {}
for target in target_columns:
    p = params.loc[target]
    y_train, y_test = merged[target].iloc[train_idx], merged[target].iloc[test_idx]

    scaler = StandardScaler(
        copy=bool(p['scaler_copy']), with_mean=bool(p['scaler_with_mean']), with_std=bool(p['scaler_with_std']))
    ridge = Ridge(
        alpha=float(p['ridge_alpha']),
        fit_intercept=bool(p['ridge_fit_intercept']),
        copy_X=bool(p['ridge_copy_X']),
        max_iter=int_or_none(p['ridge_max_iter']),
        tol=float(p['ridge_tol']),
        solver=p['ridge_solver'],
        positive=bool(p['ridge_positive']),
        random_state=int_or_none(p['ridge_random_state']),
    )

    Xs_train = scaler.fit_transform(X_train)
    Xs_test = scaler.transform(X_test)
    ridge.fit(Xs_train, y_train)
    pred_train, pred_test = ridge.predict(Xs_train), ridge.predict(Xs_test)

    # Sanity check: reproduce predictions with plain NumPy from the weights that will be saved.
    manual = ridge.intercept_ + ((X_test.to_numpy() - scaler.mean_) / scaler.scale_) @ ridge.coef_
    if not np.allclose(manual, pred_test):
        raise AssertionError(f'Manual prediction does not match sklearn for {target}')

    target_dir = MODELS_DIR / target
    target_dir.mkdir()
    joblib.dump(scaler, target_dir / 'scaler.joblib')
    joblib.dump(ridge, target_dir / 'ridge.joblib')

    row = {'intercept': float(ridge.intercept_)}
    row.update({f'{WEIGHT_PREFIX}{f}': w for f, w in zip(feature_columns, ridge.coef_)})
    row.update({f'{MEAN_PREFIX}{f}': m for f, m in zip(feature_columns, scaler.mean_)})
    row.update({f'{SCALE_PREFIX}{f}': s for f, s in zip(feature_columns, scaler.scale_)})
    row.update({
        'ridge_alpha': ridge.alpha, 'ridge_solver': ridge.solver, 'ridge_fit_intercept': ridge.fit_intercept,
        'ridge_tol': ridge.tol, 'ridge_positive': ridge.positive,
        'scaler_with_mean': scaler.with_mean, 'scaler_with_std': scaler.with_std,
        'split_test_size': SPLIT['test_size'], 'split_train_size': SPLIT['train_size'],
        'split_shuffle': SPLIT['shuffle'], 'split_random_state': SPLIT['random_state'],
        'n_features': len(feature_columns), 'n_train': len(train_idx), 'n_test': len(test_idx),
        'train_r2': r2_score(y_train, pred_train), 'test_r2': r2_score(y_test, pred_test),
        'train_mae': mean_absolute_error(y_train, pred_train), 'test_mae': mean_absolute_error(y_test, pred_test),
        'target_mean_train': y_train.mean(), 'target_mean_test': y_test.mean(),
        'sklearn_version': sklearn.__version__,
    })
    rows[target] = row

## Save feature map and metadata CSV
`feature_map.json` lists the feature order (position 0, 1, 2, ...) all models expect; inputs must be
arranged in this order before calling any saved scaler or model.

In [7]:
feature_map = {
    'description': 'Feature name -> column position expected by every scaler.joblib and ridge.joblib.',
    'n_features': len(feature_columns),
    'features': {name: i for i, name in enumerate(feature_columns)},
}
(MODELS_DIR / 'feature_map.json').write_text(json.dumps(feature_map, indent=2))

metadata_df = pd.DataFrame.from_dict(rows, orient='index')
metadata_df.index.name = 'target'
metadata_path = MODELS_DIR / 'model_metadata.csv'
metadata_df.to_csv(metadata_path)
print(f'Saved {len(metadata_df)} targets x {metadata_df.shape[1]} columns to {metadata_path}')
print(f'Built {datetime.now():%Y-%m-%d %H:%M:%S}')

Saved 21 targets x 946 columns to /home/lelandmesford/Projects/Predict_Car_Sales/models/model_metadata.csv
Built 2026-10-16 01:41:53


## Review
Metrics per target, then a round-trip check that reloads the saved joblib files and a CSV-only prediction.

In [8]:
metadata_df[['ridge_alpha', 'train_r2', 'test_r2', 'train_mae', 'test_mae', 'target_mean_train', 'target_mean_test']]

,ridge_alpha,train_r2,test_r2,train_mae,test_mae,target_mean_train,target_mean_test
target,,,,,,,
suv_crossover,800.0,0.647479,-1.725457,35579.985808,86428.723123,7.828760e+05,7.751363e+05
pickup_truck,150.0,0.871546,-0.075628,5519.984776,20578.763932,2.636090e+05,2.574976e+05
sedan,1000.0,0.621232,-5.683884,6312.924156,8348.547119,1.008095e+05,1.051467e+05
mixed_body_style,700.0,0.653926,-2.329074,5274.458631,9454.285589,9.615021e+04,9.434800e+04
minivan,200.0,0.704195,0.257021,1474.027093,3506.736544,3.400300e+04,3.329086e+04
commercial_van,400.0,0.722142,-0.277014,1693.590379,3424.605439,3.147521e+04,3.140600e+04
sports_car,1400.0,0.370339,-0.518204,995.877210,2231.682134,1.034293e+04,1.031129e+04
hatchback_liftback,600.0,0.760355,-2.033708,775.146756,1512.462983,8.430643e+03,8.649143e+03
wagon,400.0,0.629189,0.020317,221.729303,478.194088,1.845786e+03,1.813143e+03


In [9]:
# Round trip: load saved artifacts and the saved CSV, and confirm every route gives the same predictions.
saved_order = list(json.loads((MODELS_DIR / 'feature_map.json').read_text())['features'])
saved_meta = pd.read_csv(metadata_path, index_col='target')
X_check = X_test[saved_order]
for target in target_columns:
    scaler = joblib.load(MODELS_DIR / target / 'scaler.joblib')
    ridge = joblib.load(MODELS_DIR / target / 'ridge.joblib')
    via_joblib = ridge.predict(scaler.transform(X_check))

    r = saved_meta.loc[target]
    w = r[[f'{WEIGHT_PREFIX}{f}' for f in saved_order]].to_numpy(float)
    m = r[[f'{MEAN_PREFIX}{f}' for f in saved_order]].to_numpy(float)
    s = r[[f'{SCALE_PREFIX}{f}' for f in saved_order]].to_numpy(float)
    via_csv = r['intercept'] + ((X_check.to_numpy() - m) / s) @ w
    assert np.allclose(via_joblib, via_csv), target
print('Joblib and CSV-weight predictions agree for all targets.')

Joblib and CSV-weight predictions agree for all targets.
